# CS172 Computer Vision I: CAPTCHA Recognition

**Name:** Wenbo Zhao
**Student ID:** 2024533114

This notebook is the report for your implementation and experiments. Keep reusable implementation in your package and scripts. The helper functions are provided for convenience; you may modify, replace, or remove any of them to match your implementation and report, unless the cell explicitly states otherwise.

## Report checklist

- Replace every `TODO` with your own code, result, or analysis.
- Modify any helper function when its assumptions do not match your implementation.
- Execute all cells from a clean kernel and keep the outputs visible.
- Keep test data separate from model selection and hyperparameter tuning.
- Give every figure labeled axes, a title or caption, and a short interpretation.
- Export the completed notebook to PDF before submission.

## 0. Setup

In [ ]:
import io
import json
import math
import sys
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision.io import ImageReadMode, decode_image
from torchvision.transforms import v2

# Run the notebook from the repository root (the directory that contains main.ipynb).
REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "src" / "cs172").is_dir(), "start Jupyter from the repository root"
sys.path.insert(0, str(REPO_ROOT / "src"))  # no-op when the package is installed with `pip install -e .`
sys.path.insert(0, str(REPO_ROOT / "scripts"))  # reuse the helpers of scripts/train.py

from cs172.generate_captcha import generate_captcha
from cs172.models import compute_loss, create_model
from train import make_loader, set_seed  # scripts/train.py

SEED = 42  # seed of the final run; ablations use seeds 42, 43 and 44
set_seed(SEED)  # seeds Python, NumPy and PyTorch exactly as scripts/train.py does

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

## 1. Data generation

Demonstrate valid five-digit CAPTCHAs with random colors, positions, rotations, interference lines, pixel noise, and a border.

In [ ]:
# The generator lives in src/cs172/generate_captcha.py and is imported above (not reimplemented).
# scripts/generate_data.py calls it with seed = base_seed + index, so seeds 0..7 reproduce the
# first eight images of data/training100k.zip (base seed 0).
generated_samples = []
for seed in range(8):
    # defaults: 260x80, 5 digits, font 30-40 px, rotation +-30 deg, <=5 lines, 2% noise, 2-px border
    image, label = generate_captcha(seed=seed)
    generated_samples.append((np.asarray(image), label))

for image, label in generated_samples:
    assert image.shape == (80, 260, 3) and image.dtype == np.uint8
    assert len(label) == 5 and label.isdigit()
    # the 2-pixel black border is drawn last, so it must be intact on all four sides
    assert (image[:2] == 0).all() and (image[-2:] == 0).all()
    assert (image[:, :2] == 0).all() and (image[:, -2:] == 0).all()
print("labels:", [label for _, label in generated_samples])

repeat_image, _ = generate_captcha(seed=3)
print("same seed -> identical image:", np.array_equal(np.asarray(repeat_image), generated_samples[3][0]))

with zipfile.ZipFile("data/training100k.zip") as archive:
    for seed, name in enumerate(archive.namelist()[:3]):
        stored = np.asarray(Image.open(io.BytesIO(archive.read(name))).convert("RGB"))
        same = np.array_equal(stored, generated_samples[seed][0])
        print(f"seed {seed}: stored training image {name} identical to generate_captcha(seed={seed}): {same}")

In [ ]:
def show_captcha_samples(
    samples: list[tuple[np.ndarray, list[int] | str]], columns=4, title=None
):
    rows = math.ceil(len(samples) / columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(4 * columns, 2.5 * rows), squeeze=False
    )
    for axis in axes.flat:
        axis.axis("off")
    for axis, (image, label) in zip(axes.flat, samples):
        assert isinstance(image, np.ndarray)
        assert image.dtype == np.uint8 and image.ndim == 3 and image.shape[2] == 3
        axis.imshow(image)
        axis.set_title("label: " + "".join(str(value) for value in label))
        axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_captcha_samples(generated_samples, title="Generated CAPTCHA samples")

In [ ]:
# Turn the random effects on one at a time for the same label (seed 5) to show each component.
# (Disabling rotation changes the random stream, so colors/positions differ between panels.)
component_settings = {
    "plain digits (random color, size, position)": dict(rotation_angle_range=None, max_noisy_lines=0, noise_ratio=0.0),
    "+ rotation": dict(max_noisy_lines=0, noise_ratio=0.0),
    "+ interference lines": dict(noise_ratio=0.0),
    "+ 2% pixel noise (full generator)": dict(),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 2.2))
for axis, (name, overrides) in zip(axes, component_settings.items()):
    image, label = generate_captcha(seed=5, **overrides)
    axis.imshow(np.asarray(image))
    axis.set_title(f"{name}\nlabel: {label}", fontsize=9)
    axis.axis("off")
fig.suptitle("Generator components")
plt.tight_layout()
plt.show()

**Generator summary.** `generate_captcha` draws five random digits; each digit is rendered on its own transparent RGBA canvas with a random dark color (RGB channels in [0, 160]) and a font size in [30, 40] px, rotated by a random angle in [−30°, 30°] with `expand=True`, and pasted at a random position inside its own horizontal slot (the 250-px usable width is split into five 50-px slots), which keeps the digits inside the canvas and in left-to-right order. Then 0–5 random interference lines are drawn, 2% of the pixels (416 pixels) are replaced by random colors, and a 2-pixel black border is added. A local `random.Random(seed)` makes every image reproducible: the check above confirms that the stored training images are bit-identical to the regenerated ones.

## 2. Dataset and preprocessing

Verify the custom dataset, training/validation split, preprocessing, and augmentation behavior. Images must be RGB `torch.Tensor` objects shaped `[3, H, W]`; both `torch.uint8` and `torch.float32` are supported below. Floating-point images should be in `[0, 1]` for display. If your dataset returns normalized tensors, you may need to unnormalize them for visualization.

In [ ]:
IMAGE_SIZE = (80, 260)  # (height, width) used for training, validation and testing
BATCH_SIZE = 64
TRAIN_ZIP = "data/training100k.zip"
VAL_ZIP = "data/validation.zip"

# make_loader() from scripts/train.py builds ZIP_CAPTCHA datasets (src/cs172/datasets.py), which read
# images directly from the ZIP archives, and DataLoaders exactly as during training:
#   training   -> shuffle=True, drop_last=True, random augmentation (--augment)
#   validation -> shuffle=False, deterministic preprocessing only
# workers=0 keeps the notebook simple; training used 4 workers.
train_loader = make_loader(TRAIN_ZIP, IMAGE_SIZE, BATCH_SIZE, workers=0, training=True, augment=True,
                           pin_memory=False, seed=SEED)
val_loader = make_loader(VAL_ZIP, IMAGE_SIZE, BATCH_SIZE, workers=0, training=False, pin_memory=False)
train_dataset = train_loader.dataset
val_dataset = val_loader.dataset

print("training augmentation:  ", train_dataset.transform)
print("validation augmentation:", val_dataset.transform)
print("deterministic preprocessing (both splits):", train_dataset.resize, train_dataset.to_float_and_normalize)

In [ ]:
# Training/validation split: two independently generated archives with disjoint generator seeds,
# so no validation image can appear in the training set.
for name in ("training100k", "validation"):
    manifest = json.loads(Path(f"data/{name}.json").read_text())
    first, last = manifest["seed"], manifest["seed"] + manifest["num_images"] - 1
    print(f"{name:13s} {manifest['num_images']:>7,} images, generator seeds {first:,} .. {last:,}")
print(f"loaded: train {len(train_dataset):,} images ({len(train_loader)} batches/epoch), "
      f"validation {len(val_dataset):,} images ({len(val_loader)} batches)")

# Smaller training sets (ablation 6.1 and all 10k ablations) use the --num-images option:
# a seeded random subset of the 100k archive, identical for every run that uses the same seed.
subset = make_loader(TRAIN_ZIP, IMAGE_SIZE, BATCH_SIZE, workers=0, training=True, num_images=10_000,
                     pin_memory=False, seed=SEED).dataset
print(f"--num-images 10000 --seed {SEED}: {len(subset):,} images, first files {subset.img_paths[:3]}")

In [ ]:
def inspect_dataset_sample(dataset: Dataset, name: str):
    image, label = dataset[0]
    print(f"{name} size: {len(dataset):,}")
    print(f"{name}[0] image: shape={tuple(image.shape)}, dtype={image.dtype}")
    label_text = "".join(str(int(value)) for value in label)
    print(
        f"{name}[0] label: shape={tuple(label.shape)}, dtype={label.dtype}, value={label_text}"
    )
    assert isinstance(image, torch.Tensor), "expected a torch.Tensor image"
    assert image.ndim == 3 and image.shape[0] == 3, "expected an RGB [3, H, W] image"
    assert image.dtype in (
        torch.uint8,
        torch.float32,
    ), "expected uint8 or float32 image data"
    assert tuple(label.shape) == (5,), "expected a five-digit label"
    assert label.dtype == torch.long, "expected torch.long labels"


inspect_dataset_sample(train_dataset, "training dataset")
inspect_dataset_sample(val_dataset, "validation dataset")

# The datasets normalize with mean = std = 0.5 per channel (values in [-1, 1]).
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)


def show_dataset_indices(dataset: Dataset, indices: list[int], title: str, columns=4):
    rows = math.ceil(len(indices) / columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(4 * columns, 2.5 * rows), squeeze=False
    )
    for axis, index in zip(axes.flat, indices):
        image, label = dataset[index]
        assert isinstance(image, torch.Tensor)
        assert image.ndim == 3 and image.shape[0] == 3
        assert image.dtype in (torch.uint8, torch.float32)
        image = image.detach().cpu()
        if image.dtype == torch.uint8:
            image = v2.ToDtype(torch.float32, scale=True)(image)

        # Undo channel normalization for visualization.
        mean = torch.tensor(NORM_MEAN, dtype=image.dtype).view(3, 1, 1)
        std = torch.tensor(NORM_STD, dtype=image.dtype).view(3, 1, 1)
        image = (image * std + mean).clamp(0.0, 1.0)

        assert image.min().item() >= 0.0 and image.max().item() <= 1.0
        label_text = "".join(str(int(value)) for value in label)
        axis.imshow(image.permute(1, 2, 0))
        axis.set_title(f"label: {label_text}")
        axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# Repeating index 0 should reveal random training augmentation, if enabled.
show_dataset_indices(
    train_dataset, [0] * 8, "Repeated draws of one augmented training sample"
)

# Distinct validation samples should use deterministic preprocessing only.
val_indices = list(range(min(8, len(val_dataset)))) if val_dataset is not None else []
show_dataset_indices(
    val_dataset, val_indices, "Validation samples without random augmentation"
)

In [ ]:
# One minibatch from each DataLoader
images, labels = next(iter(train_loader))
print(f"train batch images: shape={tuple(images.shape)}, dtype={images.dtype}, "
      f"range=[{images.min().item():.2f}, {images.max().item():.2f}]")
print(f"train batch labels: shape={tuple(labels.shape)}, dtype={labels.dtype}")
print("first labels:", ["".join(str(int(d)) for d in row) for row in labels[:8]])
assert images.shape == (BATCH_SIZE, 3, *IMAGE_SIZE) and images.dtype == torch.float32
assert labels.shape == (BATCH_SIZE, 5) and labels.dtype == torch.long

val_images, val_labels = next(iter(val_loader))
print(f"val batch images:   shape={tuple(val_images.shape)}, dtype={val_images.dtype}, "
      f"range=[{val_images.min().item():.2f}, {val_images.max().item():.2f}]")
print(f"val batch labels:   shape={tuple(val_labels.shape)}, dtype={val_labels.dtype}")

**Dataset and preprocessing.** `ZIP_CAPTCHA` (`src/cs172/datasets.py`) lists the archive members in sorted order, opens the ZIP lazily once per worker process, decodes each member as RGB with Pillow, and parses the label from the filename (`"12345#0.png"` → `tensor([1, 2, 3, 4, 5])`, validated to have five digits, `torch.long`). The preprocessing location is the dataset, applied identically to training, validation and test data:

1. `Resize((80, 260))` (an identity for the generated 260×80 images, kept for safety);
2. *training only*: random augmentation on the `uint8` image — `RandomAffine(degrees=5, translate=(0.05, 0.02), scale=(0.95, 1.05), fill=255)` followed by `ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1)`;
3. `ToDtype(torch.float32, scale=True)` → [0, 1], then `Normalize(mean=0.5, std=0.5)` → [−1, 1].

The model therefore receives `float32` tensors of shape `[B, 3, 80, 260]` in [−1, 1]. The repeated draws of index 0 above differ slightly (shift, scale, small rotation, color), whereas the validation samples are deterministic.

## 3. Model

Verify the interface `[batch, 3, height, width] → [batch, 10 classes, 5 digits]`, and report parameters as well as computational cost. The profiler below estimates FLOPs only for operators supported by PyTorch's profiler, so state the input size and treat the result as an approximation.

In [ ]:
# Final model: TorchVision ResNet-18 whose last layer outputs 5 x 10 logits (src/cs172/models.py).
model = create_model("resnet18", image_size=IMAGE_SIZE)

MODEL_CHECK_BATCH_SIZE = 4
MODEL_INPUT_SIZE = IMAGE_SIZE  # (height, width) expected by the model
MODEL_INPUT_DTYPE = torch.float32  # the dataset already returns normalized float32 images

In [ ]:
height, width = MODEL_INPUT_SIZE
if MODEL_INPUT_DTYPE == torch.uint8:
    random_images = torch.randint(0, 256, (MODEL_CHECK_BATCH_SIZE, 3, height, width), dtype=torch.uint8)
elif MODEL_INPUT_DTYPE == torch.float32:
    random_images = torch.rand(MODEL_CHECK_BATCH_SIZE, 3, height, width, dtype=torch.float32)
else:
    raise ValueError(f"Unsupported model input dtype: {MODEL_INPUT_DTYPE}")
random_labels = torch.randint(0, 10, (MODEL_CHECK_BATCH_SIZE, 5), dtype=torch.long)

print(f"random images: {tuple(random_images.shape)}, {random_images.dtype}")
print(f"random labels: {tuple(random_labels.shape)}, {random_labels.dtype}")

model_device = next(model.parameters()).device
model.eval()
with torch.inference_mode():
    random_logits = model(random_images.to(model_device))
print(f"model output: {tuple(random_logits.shape)}, {random_logits.dtype}")
assert tuple(random_logits.shape) == (MODEL_CHECK_BATCH_SIZE, 10, 5)

In [ ]:
def compute_num_params_and_flops(model, example_images):
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    model_device = next(model.parameters()).device
    activities = [torch.profiler.ProfilerActivity.CPU]
    if model_device.type == "cuda":
        activities.append(torch.profiler.ProfilerActivity.CUDA)
    model.eval()
    with torch.inference_mode(), torch.profiler.profile(activities=activities, with_flops=True) as profile:
        model(example_images[:1].to(model_device))
    estimated_flops = sum(event.flops or 0 for event in profile.key_averages())
    return {
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "estimated_flops_per_image": estimated_flops,
    }

if model is not None:
    cost = compute_num_params_and_flops(model, random_images)
    for name, value in cost.items():
        print(f"{name}: {value:,}")

In [ ]:
# A real (normalized) training batch through the untrained model: the loss starts near ln(10) = 2.30.
model.eval()
with torch.inference_mode():
    batch_logits = model(images[:8])
print(f"real batch: {tuple(images[:8].shape)} -> logits {tuple(batch_logits.shape)}")
print(f"initial cross-entropy (compute_loss): {compute_loss(batch_logits, labels[:8]).item():.3f}")

In [ ]:
# Parameter count and FLOPs of all four implemented architectures for the same 80x260 input
# (used again in ablations 6.6 and 6.7).
example_image = torch.rand(1, 3, *IMAGE_SIZE)
architecture_costs = {}
for name in ("resnet18", "resnet34", "captcha_resnet18", "captcha_resnet34"):
    candidate = create_model(name, image_size=IMAGE_SIZE)
    head = candidate.backbone.fc if hasattr(candidate, "backbone") else candidate.classifier
    cost = compute_num_params_and_flops(candidate, example_image)
    cost["head_parameters"] = sum(p.numel() for p in head.parameters())
    architecture_costs[name] = cost

rows = ["| Model | Pooling | Parameters | Head parameters | Estimated GFLOPs / image |", "|---|---|---:|---:|---:|"]
for name, cost in architecture_costs.items():
    pooling = "flatten 512x3x9" if name.startswith("captcha") else "global average"
    rows.append(f"| `{name}` | {pooling} | {cost['total_parameters'] / 1e6:.2f} M | "
                f"{cost['head_parameters']:,} | {cost['estimated_flops_per_image'] / 1e9:.2f} |")
display(Markdown("\n".join(rows)))

**Architecture of the final model (`resnet18`).** `TorchvisionResNet` wraps `torchvision.models.resnet18(weights=None)` (trained from scratch): a 7×7/2 convolution + BN + ReLU + 3×3/2 max-pool stem, four stages of two `BasicBlock`s each (64, 128, 256, 512 channels; stages 2–4 downsample by 2 with a 1×1 projection shortcut), global average pooling, and a fully connected layer 512 → 50 that replaces the 1000-class classifier. The 50 logits are reshaped to `[batch, 10 classes, 5 digits]`; the loss (`compute_loss` in `models.py`) is the cross-entropy averaged over all five positions. For an 80×260 input, the final feature map is 512×3×9 before pooling. The model has about 11.2 M parameters (the parameter count and the profiler's FLOP estimate are printed above).

**Custom variant (`captcha_resnet18/34`).** `CAPTCHAResNet` re-implements the same stem and residual stages (`BasicBlock` with two 3×3 conv + BN layers, ReLU, and a 1×1-conv + BN projection when stride or channel count changes; Kaiming initialization as in TorchVision) but has **no global average pooling**: the 512×3×9 feature map is flattened (13,824 features) and mapped by one linear layer to 50 logits. This preserves the horizontal layout at the price of a 27× larger head (691,250 vs 25,650 parameters).

## 4. Training report

Select the best checkpoint using validation full-image accuracy, never test performance. Plot training and validation loss, per-digit accuracy, and full-image accuracy over epochs.

### Final training configuration

Run directory: `outputs/captcha100k-resnet18-aug/20261021-140309-9575a96d` (training command in `scripts/train.sh`, "Default").

| Setting | Value |
|---|---|
| Seed | 42 (`set_seed` seeds Python, NumPy and PyTorch); data generation seeds: training 0–99,999, validation 10,000,000–10,004,999 |
| Model | `resnet18` — TorchVision ResNet-18 from scratch, global average pooling, FC 512 → 5×10 |
| Training images | 100,000 (`data/training100k.zip`); validation: 5,000 (`data/validation.zip`) |
| Optimizer | AdamW (β = (0.9, 0.999)) |
| Batch size | 64 (shuffled, `drop_last=True`; 1,562 updates per epoch) |
| Initial learning rate and schedule | 1e-3, cosine annealing to 0 over 20 epochs (`CosineAnnealingLR`, stepped once per epoch), no warm-up |
| Weight decay | 1e-4 (decoupled, AdamW) |
| Data augmentation | `RandomAffine(degrees=5, translate=(0.05, 0.02), scale=(0.95, 1.05), fill=255)` + `ColorJitter(0.1, 0.1, 0.1)` (training only) |
| Epochs and best-checkpoint epoch | 20 epochs; best checkpoint at epoch 16 (validation full-image accuracy 100.00%, selected by validation full-image accuracy) |
| Hardware / time | one Tesla V100-PCIE-16GB, ≈ 33 min of training in total |

In [ ]:
FINAL_RUN_DIR = Path("outputs/captcha100k-resnet18-aug/20261021-140309-9575a96d")
final_config = json.loads((FINAL_RUN_DIR / "config.json").read_text())
final_metrics = json.loads((FINAL_RUN_DIR / "metrics.json").read_text())

# The table above is copied from the files that were saved with the run.
for key in ("model", "data_dir", "num_images", "val_data_dir", "image_size", "epochs", "batch_size",
            "learning_rate", "weight_decay", "augment", "seed"):
    print(f"{key:14s} {final_config[key]}")
best_epoch = final_metrics[-1]["best_epoch"]
print(f"best epoch: {best_epoch}, best validation full-image accuracy: {final_metrics[-1]['best_val_image_acc']:.2%}")
print(f"total training time: {sum(r['train_time_sec'] for r in final_metrics) / 60:.1f} min")

### TensorBoard curve reader

The functions below read scalar summaries directly from a TensorBoard log directory. The tag mapping must match the names recorded by your training pipeline. TensorBoard is optional and can be installed with `pip install tensorboard`. 

Leveraging Tensorboard to plot curves is just an example. **You can save the metrics in any format you prefer, as long as you can plot them.**

In [ ]:
def read_tensorboard_scalars(log_dir):
    try:
        from tensorboard.backend.event_processing.event_accumulator import (
            EventAccumulator,
        )
    except ImportError as error:
        raise ImportError("Install TensorBoard to read event files.") from error

    accumulator = EventAccumulator(str(log_dir))
    accumulator.Reload()
    history = {}
    for tag in accumulator.Tags().get("scalars", []):
        events = accumulator.Scalars(tag)
        history[tag] = {
            "step": [event.step for event in events],
            "value": [event.value for event in events],
        }
    return history


def read_metrics_json(run_dir):
    """Same format as read_tensorboard_scalars(), built from the metrics.json written by scripts/train.py."""
    history = {}
    for record in json.loads((Path(run_dir) / "metrics.json").read_text()):
        for split in ("train", "val"):
            for name, value in record[split].items():
                series = history.setdefault(f"{split}/{name}", {"step": [], "value": []})
                series["step"].append(record["epoch"])
                series["value"].append(value)
    return history


def plot_training_curves(history, tag_map):
    metrics = ("loss", "digit_acc", "image_acc")
    titles = ("Loss (log scale)", "Per-digit accuracy", "Full-image accuracy")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    for axis, metric, title in zip(axes, metrics, titles):
        for split in ("train", "val"):
            tag = tag_map[f"{split}_{metric}"]
            series = history[tag]
            axis.plot(series["step"], series["value"], marker="o", markersize=3, label=split)
        if metric == "loss":
            axis.set_yscale("log")
        else:
            axis.set_ylim(0.98, 1.0005)  # zoom: both splits exceed 98% from epoch 2 on
        axis.set(xlabel="epoch", title=title)
        axis.grid(alpha=0.25)
        axis.legend()
    fig.suptitle("Final run (ResNet-18, 100k images): training vs. validation")
    plt.show()

In [ ]:
# Scalars were logged by scripts/train.py --tensorboard with these tags (step = epoch).
TENSORBOARD_LOG_DIR = FINAL_RUN_DIR / "tensorboard"
TENSORBOARD_TAGS = {
    "train_loss": "train/loss",
    "val_loss": "val/loss",
    "train_digit_acc": "train/digit_acc",
    "val_digit_acc": "val/digit_acc",
    "train_image_acc": "train/image_acc",
    "val_image_acc": "val/image_acc",
}

try:
    final_history = read_tensorboard_scalars(TENSORBOARD_LOG_DIR)
except ImportError:  # TensorBoard is optional; metrics.json holds the same values
    final_history = read_metrics_json(FINAL_RUN_DIR)
print("Available scalar tags:", sorted(final_history))
plot_training_curves(final_history, TENSORBOARD_TAGS)

print("epoch | train loss  img acc | val loss   img acc")
for record in final_metrics:
    print(f"{record['epoch']:5d} | {record['train']['loss']:.4f}  {record['train']['image_acc']:7.2%} | "
          f"{record['val']['loss']:.4f}  {record['val']['image_acc']:7.2%}")

### Training analysis

- **Convergence.** With 100k images (1,562 updates per epoch) the model converges very quickly: after the first epoch the validation full-image accuracy is already 98.92%, it reaches 99.9% at epoch 4, and from epoch 12 on it stays between 99.96% and 100%. The training loss decreases almost monotonically from 0.25 to 2·10⁻⁴.
- **Stability.** Early epochs are noisy on the validation set (e.g. a dip to 99.22% at epoch 6, while the learning rate is still close to 1e-3); as the cosine schedule lowers the learning rate, the validation curves become flat and the loss falls below 10⁻³.
- **Best checkpoint.** Epoch 16 is the first epoch that reaches 100.00% validation full-image accuracy (5,000/5,000); later epochs reach 99.98–100%, so `best.pt` is the epoch-16 model (selection uses validation accuracy only).
- **Train–validation gap.** There is no overfitting on the validation distribution: the training accuracy (measured in train mode on *augmented* images) is even slightly below the validation accuracy (e.g. 99.95% vs. 100% at epoch 16), and both losses decrease together. As Section 5 shows, the real gap is between the generated validation data and the provided test data, i.e. a distribution shift rather than classic overfitting.

## 5. Performance evaluation

Evaluate the checkpoint selected by validation full-image accuracy exactly once on the five numeric test sets under `data/test/` (500 images in total). Concatenate the test sets and report aggregate per-digit and full-image accuracy.

In [ ]:
# Load the checkpoint selected by validation full-image accuracy (epoch 16 of the final run).
checkpoint = torch.load(FINAL_RUN_DIR / "checkpoints" / "best.pt", map_location="cpu")
test_model = create_model(final_config["model"], image_size=tuple(final_config["image_size"]))
test_model.load_state_dict(checkpoint["model"])
print(f"loaded best.pt: epoch {checkpoint['epoch'] + 1}, "
      f"validation full-image accuracy {checkpoint['best_image_acc']:.2%}")

# The deterministic preprocessing used for validation, taken from the validation dataset itself:
# Resize((80, 260)) -> ToDtype(float32, scale=True) -> Normalize(mean=0.5, std=0.5)
test_transform = v2.Compose([val_dataset.resize, val_dataset.to_float_and_normalize])

TEST_DIRS = [
    Path("data/test/test0"),
    Path("data/test/test1"),
    Path("data/test/test2"),
    Path("data/test/test3"),
    Path("data/test/test4"),
]
EVAL_BATCH_SIZE = 100

### Evaluation helpers

This small evaluation-only `Dataset` loads RGB images, parses labels from filenames, and applies deterministic preprocessing. The five datasets are concatenated below so the reported metrics cover all 500 test images.

In [ ]:
# CAUTION: DO NOT modify the code below. It will be used to evaluate your model on the hidden test set.

class EvaluationDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_paths = sorted(
            path for path in Path(image_dir).iterdir()
            if path.suffix.lower() in {".png", ".jpg", ".jpeg"}
        )
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        image = decode_image(str(image_path), mode=ImageReadMode.RGB)
        label_text = image_path.stem.split("#", maxsplit=1)[0]
        label = torch.tensor([int(digit) for digit in label_text])
        if self.transform is not None:
            image = self.transform(image)
        return image, label

@torch.inference_mode()
def evaluate_accuracy(model: torch.nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    correct_digits = 0
    correct_images = 0
    num_digits = 0
    num_images = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        logits = model(images)
        assert logits.shape == (targets.shape[0], 10, 5)
        predictions = logits.argmax(dim=1)
        correct = predictions.eq(targets)
        correct_digits += correct.sum().item()
        correct_images += correct.all(dim=1).sum().item()
        num_digits += targets.numel()
        num_images += targets.shape[0]

    digit_accuracy = correct_digits / num_digits
    image_accuracy = correct_images / num_images
    score = (
        20 * max(digit_accuracy - 0.95, 0.0) / 0.05
        + 20 * max(image_accuracy - 0.90, 0.0) / 0.10
    )
    return {
        "num_images": num_images,
        "digit_acc": digit_accuracy,
        "image_acc": image_accuracy,
        "score": score,
    }

In [ ]:
test_datasets = {
    path.name: EvaluationDataset(path, test_transform) for path in TEST_DIRS
}
combined_test_dataset = ConcatDataset(list(test_datasets.values()))
assert len(combined_test_dataset) == 500
test_loader = DataLoader(
    combined_test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False
)

# Run the final evaluation after connecting the selected model and preprocessing.
test_model = test_model.to(DEVICE)
test_result = evaluate_accuracy(test_model, test_loader, DEVICE)
print(f"images:              {test_result['num_images']}")
print(f"per-digit accuracy:  {test_result['digit_acc']:.2%}")
print(f"full-image accuracy: {test_result['image_acc']:.2%}")
print(f"evaluation score:    {test_result['score']:.2f}")

In [ ]:
# Breakdown of the same evaluation by test set (each set has 100 images).
per_set_results = {}
for name, dataset in test_datasets.items():
    loader = DataLoader(dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)
    per_set_results[name] = evaluate_accuracy(test_model, loader, DEVICE)

rows = ["| Test set | Images | Per-digit accuracy | Full-image accuracy |", "|---|---:|---:|---:|"]
for name, result in per_set_results.items():
    rows.append(f"| {name} | {result['num_images']} | {result['digit_acc']:.2%} | {result['image_acc']:.2%} |")
rows.append(f"| **all** | **{test_result['num_images']}** | **{test_result['digit_acc']:.2%}** | "
            f"**{test_result['image_acc']:.2%}** |")
display(Markdown("\n".join(rows)))

**Result.** The selected ResNet-18 reaches about 89.5% per-digit and 74% full-image accuracy on the 500 test images (the exact values are printed above; `scripts/evaluate.py` gave 89.52% / 74.00% for the same checkpoint). Both are below the thresholds of the score formula (95% / 90%), so the evaluation score is 0, although the same model is ≈100% correct on 5,000 generated validation images. The breakdown shows that the errors are concentrated in `test0`–`test3` (≈67–73% full-image accuracy), while `test4`, whose images look most like our generated data (rotated digits, interference lines and pixel noise), reaches ≈90%. Section 6.8 analyses the failures; the main cause is a mismatch between the digit layout of our generator and that of the test images.

## 6. Analysis and ablations

For every comparison, change only the named factor and keep unrelated settings fixed. Select each run's checkpoint using validation full-image accuracy. When showing curves over epochs, plot validation metrics. When comparing final performance, report the aggregate test results directly. Use simple plots or tables and state a concise conclusion supported by the results.

**Protocol shared by all ablations.** Unless a factor is varied, every run uses the "baseline-10k" setting: TorchVision ResNet-18, a 10,000-image seeded subset of `data/training100k.zip` (`--num-images 10000`), augmentation on, AdamW, batch size 64, learning rate 1e-3 with cosine annealing, weight decay 1e-4, 20 epochs, image size 80×260. Every setting was trained with **three seeds (42, 43, 44)**; the seed also selects the 10k subset, so the subsets are identical across settings with the same seed. For every run, `best.pt` is the epoch with the highest validation full-image accuracy, and the aggregate test accuracy over all 500 test images was computed with `scripts/evaluate.py` and saved in the run's `evaluation.json`. Tables report mean ± standard deviation over the three seeds; curves show the seed mean with the min–max range shaded. "Train img acc @best" is the training full-image accuracy logged at the selected epoch (on augmented images when augmentation is on).

In [ ]:
OUTPUTS_DIR = Path("outputs")
_RUN_CACHE = {}


def load_runs(experiment):
    """Load config.json, metrics.json and evaluation.json of every seed of one experiment."""
    if experiment in _RUN_CACHE:
        return _RUN_CACHE[experiment]
    runs = []
    for config_path in sorted((OUTPUTS_DIR / experiment).glob("*/config.json")):
        run_dir = config_path.parent
        config = json.loads(config_path.read_text())
        history = json.loads((run_dir / "metrics.json").read_text())
        results = json.loads((run_dir / "evaluation.json").read_text())["results"]
        assert len(results) == 5, f"{run_dir} was not evaluated on the five test sets"
        num_test = sum(result["num_images"] for result in results.values())
        best_epoch = history[-1]["best_epoch"]
        best_record = history[best_epoch - 1]
        assert best_record["epoch"] == best_epoch
        events = sorted((run_dir / "tensorboard").glob("events.out.tfevents.*"))
        runs.append({
            "config": config,
            "history": history,
            "seed": config["seed"],
            "best_epoch": best_epoch,
            "train_image_acc": best_record["train"]["image_acc"],
            "val_image_acc": best_record["val"]["image_acc"],
            "final_train_loss": history[-1]["train"]["loss"],
            "final_val_loss": history[-1]["val"]["loss"],
            "test_digit_acc": sum(r["digit_acc"] * r["num_images"] for r in results.values()) / num_test,
            "test_image_acc": sum(r["image_acc"] * r["num_images"] for r in results.values()) / num_test,
            "num_test_images": num_test,
            "train_time": sum(record["train_time_sec"] for record in history),
            # TensorBoard event files are named events.out.tfevents.<time>.<host>.<pid>.0
            "node": events[0].name.split(".")[4] if events else "?",
        })
    assert len(runs) > 0, f"no runs found in {OUTPUTS_DIR / experiment}"
    assert all(run["num_test_images"] == 500 for run in runs)
    _RUN_CACHE[experiment] = runs
    return runs


def mean_std(runs, key, scale=100.0):
    values = np.array([run[key] for run in runs], dtype=float) * scale
    return values.mean(), values.std()


def fmt(runs, key, scale=100.0, digits=2):
    mean, std = mean_std(runs, key, scale)
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


def summary_table(experiments, extra_columns=None):
    """experiments: {setting label: experiment directory}; extra_columns: {name: fn(label, runs) -> str}."""
    extra_columns = extra_columns or {}
    header = ["Setting", "Seeds", "Best epochs", "Train img acc @best (%)", "Val img acc @best (%)",
              "Final train loss", "Final val loss", *extra_columns, "**Test digit acc (%)**", "**Test img acc (%)**"]
    rows = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for label, experiment in experiments.items():
        runs = load_runs(experiment)
        cells = [str(label), ", ".join(str(r["seed"]) for r in runs), ", ".join(str(r["best_epoch"]) for r in runs),
                 fmt(runs, "train_image_acc"), fmt(runs, "val_image_acc"),
                 f"{np.mean([r['final_train_loss'] for r in runs]):.1e}",
                 f"{np.mean([r['final_val_loss'] for r in runs]):.1e}",
                 *[fn(label, runs) for fn in extra_columns.values()],
                 f"**{fmt(runs, 'test_digit_acc')}**", f"**{fmt(runs, 'test_image_acc')}**"]
        rows.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(rows)))


def plot_val_curves(experiments, title):
    """Validation loss and full-image accuracy over epochs (mean over seeds, min-max shaded)."""
    panels = (("loss", "Validation loss (log scale)", None),
              ("image_acc", "Validation full-image accuracy", None),
              ("image_acc", "Validation full-image accuracy (zoom)", (0.98, 1.0005)))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    for label, experiment in experiments.items():
        runs = load_runs(experiment)
        epochs = np.array([record["epoch"] for record in runs[0]["history"]])
        for axis, (metric, _, _) in zip(axes, panels):
            values = np.array([[record["val"][metric] for record in run["history"]] for run in runs])
            (line,) = axis.plot(epochs, values.mean(axis=0), marker="o", markersize=3, label=str(label))
            if len(runs) > 1:
                axis.fill_between(epochs, values.min(axis=0), values.max(axis=0), color=line.get_color(), alpha=0.15)
    for axis, (metric, panel_title, ylim) in zip(axes, panels):
        axis.set(xlabel="epoch", title=panel_title)
        if metric == "loss":
            axis.set_yscale("log")
        if ylim is not None:
            axis.set_ylim(*ylim)
        axis.grid(alpha=0.25)
    axes[1].legend()
    fig.suptitle(f"{title}: validation curves (mean over 3 seeds, shaded = min–max)")
    plt.show()


def plot_test_bars(experiments, title, xlabel):
    """Aggregate test accuracy (500 images) per setting, mean ± std over seeds."""
    labels = [str(label) for label in experiments]
    x = np.arange(len(labels))
    fig, axis = plt.subplots(figsize=(1.6 * len(labels) + 3, 3.8), constrained_layout=True)
    for offset, key, name in ((-0.2, "test_digit_acc", "per-digit"), (0.2, "test_image_acc", "full-image")):
        stats = [mean_std(load_runs(experiment), key) for experiment in experiments.values()]
        means, stds = np.array(stats).T
        bars = axis.bar(x + offset, means, width=0.4, yerr=stds, capsize=4, label=f"test {name} accuracy")
        axis.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)
    axis.axhline(95, color="gray", linestyle="--", linewidth=1)
    axis.axhline(90, color="gray", linestyle=":", linewidth=1)
    axis.text(len(labels) - 0.5, 95.5, "95% / 90% score thresholds", ha="right", fontsize=8, color="gray")
    axis.set_xticks(x, labels)
    axis.set(xlabel=xlabel, ylabel="accuracy (%)", ylim=(40, 100), title=f"{title}: aggregate test accuracy (500 images)")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(loc="lower right")
    plt.show()

### 6.1 Training-set size

Train with at least three training-set sizes while keeping the other settings fixed.

**Controlled settings.** Baseline protocol (ResNet-18, augmentation, AdamW, batch 64, lr 1e-3 cosine, weight decay 1e-4, 20 epochs, seeds 42–44); only `--num-images` changes: 1k, 2k, 5k, 10k, 20k, 50k and the full 100k archive (the 100k runs are the final configuration). Note that the number of epochs is fixed, so smaller sets also receive fewer gradient updates (300 for 1k vs. 31,240 for 100k).

In [ ]:
SIZE_EXPERIMENTS = {
    1_000: "size-1000",
    2_000: "size-2000",
    5_000: "size-5000",
    10_000: "baseline-10k",
    20_000: "size-20000",
    50_000: "size-50000",
    100_000: "captcha100k-resnet18-aug",
}
sizes = np.array(list(SIZE_EXPERIMENTS))
fig, axis = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
for key, name, style in (("test_digit_acc", "test per-digit", "o-"), ("test_image_acc", "test full-image", "s-"),
                         ("val_image_acc", "validation full-image (best ckpt)", "^--")):
    stats = np.array([mean_std(load_runs(experiment), key) for experiment in SIZE_EXPERIMENTS.values()])
    axis.errorbar(sizes, stats[:, 0], yerr=stats[:, 1], fmt=style, capsize=4, label=name)
axis.axhline(95, color="gray", linestyle="--", linewidth=1)
axis.axhline(90, color="gray", linestyle=":", linewidth=1)
axis.set_xscale("log")
axis.set_xticks(sizes, [f"{n // 1000}k" for n in sizes])
axis.set(xlabel="number of training images (log scale)", ylabel="accuracy (%)",
         title="Accuracy vs. training-set size (mean ± std over 3 seeds)")
axis.grid(alpha=0.25)
axis.legend()
plt.show()

summary_table({f"{n:,}": experiment for n, experiment in SIZE_EXPERIMENTS.items()})

**Conclusion.** Performance increases steeply from 1k to 5k images and then saturates. With 1k images (300 updates) training has not converged (validation full-image accuracy 70.6 ± 14.6%, one seed only 50%) and test full-image accuracy is only 29.9%; 2k → 62.5%, 5k → 68.3%, 10k → 70.9%. Beyond 10k the validation accuracy is ≥ 99.9% and the test full-image accuracy stays at ≈ 70–72% (20k: 69.6%, 50k: 72.2%, 100k: 72.4%), i.e. a 10× larger data set gains only ≈ 1.5 points, within about one standard deviation. More images drawn from the *same* generator cannot close the gap to the test distribution, because the test errors are caused by a distribution shift (Section 6.8), not by too little data.

### 6.2 Data augmentation and overfitting

Compare matched runs with and without data augmentation.

**Controlled settings.** Baseline protocol with 10k images; only `--augment` changes (with: `RandomAffine` ±5°, ±5%/±2% translation, 0.95–1.05 scale + `ColorJitter` 0.1; without: deterministic preprocessing only). Seeds 42–44, so both settings use the same three 10k subsets.

In [ ]:
AUG_EXPERIMENTS = {"with augmentation": "baseline-10k", "without augmentation": "noaug-10k"}
plot_val_curves(AUG_EXPERIMENTS, "Data augmentation")
summary_table(AUG_EXPERIMENTS)
plot_test_bars(AUG_EXPERIMENTS, "Data augmentation", "training setting")

**Conclusion.** Without augmentation the network fits the training set perfectly (training full-image accuracy 100% in the last epochs, final training loss ≈ 4·10⁻⁴) while the validation accuracy saturates at a lower level (99.73% vs. 99.89%) with a higher final validation loss (≈ 2.6·10⁻³ vs. 1.1·10⁻³) and an earlier best epoch (7–16) — the classic signature of overfitting. With augmentation the training accuracy stays slightly *below* the validation accuracy, so the train–validation gap disappears. The effect on the test set is much larger than on the validation set: augmentation raises the aggregate test full-image accuracy from 55.7 ± 1.1% to 70.9 ± 1.1% (per-digit 82.9% → 89.3%), the largest gain of all ablations. The random translation/scale/rotation moves digits slightly away from the positions produced by the generator, which makes the model less dependent on the exact training layout and thus more robust to the shifted test layout.

### 6.3 Regularization and overfitting

Compare multiple weight-decay values while keeping the model and all other training settings, including augmentation, fixed.

**Controlled settings.** Baseline protocol with 10k images and augmentation; only the (decoupled AdamW) weight decay changes: 0, 1e-4 (baseline), 1e-2, 1e-1. Seeds 42–44.

In [ ]:
WD_EXPERIMENTS = {"wd = 0": "wd-0", "wd = 1e-4 (baseline)": "baseline-10k", "wd = 1e-2": "wd-0.01", "wd = 1e-1": "wd-0.1"}
plot_val_curves(WD_EXPERIMENTS, "Weight decay")
summary_table(WD_EXPERIMENTS)
plot_test_bars(WD_EXPERIMENTS, "Weight decay", "weight decay")

**Conclusion.** Weight decay has no measurable effect in this range. All four settings reach ≈ 99.9% validation full-image accuracy with training accuracy ≈ validation accuracy, and the test full-image accuracy is 70.3 ± 1.6% (0), 70.9 ± 1.1% (1e-4), 70.0 ± 0.2% (1e-2) and 71.3 ± 1.7% (1e-1) — differences far below one standard deviation. Two reasons: (i) with augmentation there is almost no overfitting left to regularize (Section 6.2), and (ii) with AdamW the decay is decoupled and scaled by the cosine-annealed learning rate, while the BN-normalized convolution weights are scale-invariant, so weight decay mainly changes the effective learning rate rather than the function class. Weight decay does not address the train/test distribution shift, which dominates the test error.

### 6.4 Batch size

Compare multiple batch sizes while keeping the model, data, optimizer, and learning-rate schedule fixed. Measure training time consistently on the same hardware.

**Controlled settings.** Baseline protocol with 10k images; only `--batch-size` changes (16, 32, 64, 128, 256) with the same learning rate 1e-3, cosine schedule and 20 epochs (so larger batches perform fewer updates: 625 vs. 39 per epoch). These runs (`outputs/bsx*-10k`) were made specifically for this comparison: for every seed, all five batch sizes ran on the same GPU node (seeds 42 and 43 on `sist_gpu67`, seed 44 on `sist_gpu58`), with 4 data-loading workers. Training time is the sum of the training-loop time over the 20 epochs (validation excluded), as recorded by `scripts/train.py`. (The earlier single-seed runs `outputs/bs*-10k` are not used because they ran on different nodes.)

In [ ]:
BATCH_EXPERIMENTS = {16: "bsx16-10k", 32: "bsx32-10k", 64: "bsx64-10k", 128: "bsx128-10k", 256: "bsx256-10k"}
plot_val_curves(BATCH_EXPERIMENTS, "Batch size")
summary_table(
    BATCH_EXPERIMENTS,
    extra_columns={
        "Updates / epoch": lambda bs, runs: f"{10_000 // bs}",
        "Val img acc @ epoch 3 (%)": lambda bs, runs: f"{np.mean([r['history'][2]['val']['image_acc'] for r in runs]) * 100:.1f}",
        "Train time (s)": lambda bs, runs: fmt(runs, "train_time", scale=1.0, digits=0),
    },
)
plot_test_bars(BATCH_EXPERIMENTS, "Batch size", "batch size")

# Training time per seed / node (same node within a column)
seeds = [run["seed"] for run in load_runs(BATCH_EXPERIMENTS[64])]
nodes = [run["node"] for run in load_runs(BATCH_EXPERIMENTS[64])]
rows = ["| Batch size | " + " | ".join(f"seed {s} ({n})" for s, n in zip(seeds, nodes)) + " |",
        "|---:|" + "---:|" * len(seeds)]
for batch_size, experiment in BATCH_EXPERIMENTS.items():
    runs = {run["seed"]: run for run in load_runs(experiment)}
    assert [runs[s]["node"] for s in seeds] == nodes, "timings must come from the same node per seed"
    rows.append(f"| {batch_size} | " + " | ".join(f"{runs[s]['train_time']:.0f} s" for s in seeds) + " |")
display(Markdown("**Training time for 20 epochs (s)**\n\n" + "\n".join(rows)))

**Conclusion.**
- **Convergence.** With a fixed learning rate and number of epochs, small batches converge much faster *per epoch* because they perform more updates: after 3 epochs batch 16 reaches ≈ 97% validation full-image accuracy, batch 64 ≈ 96%, while batch 128 and 256 are still at ≈ 75% and ≈ 39%. By the end of training all batch sizes reach ≥ 99.8% validation accuracy; batch 256 is slightly worse (99.81%, highest final validation loss ≈ 3.7·10⁻³) because 780 updates in total are not quite enough.
- **Accuracy.** The aggregate test accuracies are statistically indistinguishable (full-image 68.7–71.4%, per-digit 87.9–89.1%, standard deviations 0.3–2.1 points); batch 256 has the largest seed-to-seed variance.
- **Speed.** Training time does not shrink proportionally with larger batches: batch 32 and 64 are fastest (≈ 170 s), batch 16 is slowest (≈ 220 s, per-step overhead of 625 small steps), and batch 128/256 are again slower (≈ 180 s / ≈ 210 s). With only 4 data-loading workers decoding PNGs from a ZIP archive, the input pipeline — not the GPU — limits throughput for ResNet-18 on 80×260 images, so larger batches bring no speed-up. (The nodes are shared cluster machines, so other jobs may add some noise to the timings; the pattern — batch 16 slowest, 32/64 fastest, 128 and 256 in between — is, however, the same for all three seeds.)
- **Trade-off.** Batch 32–64 gives the best combination of fast wall-clock training, fast convergence and stable results at lr 1e-3; larger batches would need a larger learning rate or more epochs.

### 6.5 Learning rate

Compare multiple initial learning rates while keeping the optimizer and learning-rate schedule otherwise fixed.

**Controlled settings.** Baseline protocol with 10k images; AdamW with cosine annealing over 20 epochs; only the initial learning rate changes: 1e-4, 3e-4, 1e-3 (baseline), 3e-3, 1e-2. Seeds 42–44.

In [ ]:
LR_EXPERIMENTS = {"1e-4": "lr-0.0001", "3e-4": "lr-0.0003", "1e-3 (baseline)": "baseline-10k", "3e-3": "lr-0.003", "1e-2": "lr-0.01"}
plot_val_curves(LR_EXPERIMENTS, "Learning rate")
summary_table(
    LR_EXPERIMENTS,
    extra_columns={
        "Val img acc @ epoch 5 (%)": lambda lr, runs: fmt([{"v": r["history"][4]["val"]["image_acc"]} for r in runs], "v", digits=1),
    },
)
plot_test_bars(LR_EXPERIMENTS, "Learning rate", "initial learning rate")

**Conclusion.**
- **Convergence speed.** A small learning rate converges slowly: with 1e-4 the validation full-image accuracy is only ≈ 75% after 5 epochs and ends at 99.0% (training accuracy also only 99.0%, i.e. underfitting within 20 epochs). 3e-4 and 1e-3 converge fastest and most smoothly.
- **Stability.** Large learning rates make the early epochs unstable: with 1e-2 one seed stays at 0% validation full-image accuracy for 7 epochs (epoch-5 accuracy 63 ± 45% over seeds) and another drops from 99.2% to 71.2% between epochs 9 and 10; with 3e-3 the validation accuracy also jumps back and forth by several points. Both only settle once the cosine schedule has decayed the learning rate, and then reach ≈ 99.8–99.9% validation accuracy.
- **Test performance.** 1e-3 is best (70.9 ± 1.1% full-image, 89.3% per-digit), followed by 3e-4 (69.1%); both extremes are clearly worse: 1e-4 → 63.8%, 3e-3 → 66.1%, 1e-2 → 61.5 ± 5.4% (largest variance). Interestingly, 3e-3 and 1e-2 reach the same validation accuracy as 1e-3 but generalize worse to the shifted test data, so validation accuracy alone does not reveal this difference. 1e-3 is a good default for AdamW here.

### 6.6 Backbone comparison

Compare at least two backbones (e.g., ResNet-18 and ResNet-34) under comparable settings.

**Controlled settings.** Baseline protocol with 10k images (augmentation, AdamW, batch 64, lr 1e-3 cosine, weight decay 1e-4, 20 epochs, seeds 42–44); only the depth changes: ResNet-18 (BasicBlocks 2-2-2-2) vs. ResNet-34 (3-4-6-3), both for the TorchVision variant with global average pooling and for our flattening `CAPTCHAResNet`. Parameters and FLOPs are those computed in Section 3 for one 80×260 image.

In [ ]:
BACKBONE_EXPERIMENTS = {
    "ResNet-18 (GAP)": ("resnet18", "baseline-10k"),
    "ResNet-34 (GAP)": ("resnet34", "resnet34-10k"),
    "CAPTCHA-ResNet-18 (flatten)": ("captcha_resnet18", "captcha18-10k"),
    "CAPTCHA-ResNet-34 (flatten)": ("captcha_resnet34", "captcha34-10k"),
}
cost_columns = {
    "Params (M)": lambda label, runs: f"{architecture_costs[BACKBONE_EXPERIMENTS[label][0]]['total_parameters'] / 1e6:.2f}",
    "GFLOPs": lambda label, runs: f"{architecture_costs[BACKBONE_EXPERIMENTS[label][0]]['estimated_flops_per_image'] / 1e9:.2f}",
}
backbone_dirs = {label: experiment for label, (_, experiment) in BACKBONE_EXPERIMENTS.items()}
plot_val_curves({k: backbone_dirs[k] for k in ("ResNet-18 (GAP)", "ResNet-34 (GAP)")}, "Backbone depth (TorchVision, GAP)")
summary_table(backbone_dirs, extra_columns=cost_columns)
plot_test_bars(backbone_dirs, "Backbone", "model")

**Conclusion.** Additional depth does not help. ResNet-34 has ≈ 1.9× the parameters (21.3 M vs. 11.2 M) and ≈ 2× the FLOPs of ResNet-18, reaches the same ≈ 99.9% validation accuracy, but is worse on the test set: 67.2 ± 1.7% vs. 70.9 ± 1.1% full-image accuracy (86.6% vs. 89.3% per-digit). For the flattening CAPTCHA-ResNet the deeper network is not better either (62.0 ± 1.5% vs. 60.1 ± 1.2%, within noise) and converges more slowly (its validation accuracy is still ≈ 9% after 5 epochs). Recognizing five clean digits is a simple task for ResNet-18 already; the extra capacity mainly fits the generator-specific details more closely, which does not transfer to the shifted test images, and it doubles the cost. ResNet-18 is therefore the better backbone.

### 6.7 Global average pooling

Compare global average pooling with flattening the final spatial feature map while keeping the backbone and training settings comparable.

**Controlled settings.** Same stem, residual stages (ResNet-18 depth), Kaiming initialization and training protocol (10k images, augmentation, AdamW, batch 64, lr 1e-3 cosine, weight decay 1e-4, 20 epochs, seeds 42–44); only the head differs: TorchVision `resnet18` (global average pooling → FC 512 → 50) vs. our `captcha_resnet18` (flatten the 512×3×9 map → FC 13,824 → 50). The ResNet-34 pair is listed as a second comparison.

In [ ]:
POOLING_EXPERIMENTS = {
    "ResNet-18 + GAP": ("resnet18", "baseline-10k"),
    "ResNet-18 + flatten (no GAP)": ("captcha_resnet18", "captcha18-10k"),
    "ResNet-34 + GAP": ("resnet34", "resnet34-10k"),
    "ResNet-34 + flatten (no GAP)": ("captcha_resnet34", "captcha34-10k"),
}
pooling_dirs = {label: experiment for label, (_, experiment) in POOLING_EXPERIMENTS.items()}
pooling_columns = {
    "Params (M)": lambda label, runs: f"{architecture_costs[POOLING_EXPERIMENTS[label][0]]['total_parameters'] / 1e6:.2f}",
    "Head params": lambda label, runs: f"{architecture_costs[POOLING_EXPERIMENTS[label][0]]['head_parameters']:,}",
}
plot_val_curves({k: pooling_dirs[k] for k in ("ResNet-18 + GAP", "ResNet-18 + flatten (no GAP)")}, "Global average pooling (ResNet-18)")
summary_table(pooling_dirs, extra_columns=pooling_columns)
plot_test_bars(pooling_dirs, "Global average pooling", "model")

**Conclusion.** Preserving the spatial map did **not** help in our setting. On the generated validation data both heads are almost perfect (99.89% with GAP vs. 99.81% without), but the flattening head trains more slowly (≈ 13% validation accuracy after 3 epochs vs. ≈ 98% with GAP), and its training accuracy at the selected checkpoint is lower (99.71% vs. 99.90%). On the test set it is clearly worse: 60.1 ± 1.2% vs. 70.9 ± 1.1% full-image accuracy (83.6% vs. 89.3% per-digit); the same holds for the ResNet-34 pair (62.0% vs. 67.2%).

*Spatial information.* GAP does not remove the information that is needed: after five stride-2 stages each of the 3×9 output cells has a theoretical receptive field (≈ 435 px) larger than the whole 80×260 image, and zero padding at the image borders lets the network encode *where* a digit is in its channels, so the 512-d pooled vector can still distinguish the five positions. The flatten head instead assigns separate weights to every (cell, position) pair, so each output position learns to look at fixed cells — exactly the 50-px slots of our generator.

*Parameter cost and overfitting risk.* Flattening multiplies the head size by 27 (691,250 vs. 25,650 parameters; 11.87 M vs. 11.20 M in total) and ties the classifier to absolute positions. This is harmless on in-distribution validation data but becomes a liability when test digits appear in other places (Section 6.8): the position-specific head overfits to the training layout, while the pooled representation is more translation-tolerant. Flattening could pay off if the training data covered the test layouts (or with a sequence head such as CTC/attention), but with the current generator GAP generalizes better.

### 6.8 Prediction visualization

Show a small set of test predictions beside their ground-truth labels. If full-image accuracy is below 100%, include and analyze failure cases.

The final model (`test_model`, Section 5) is run once more on the 500 test images; the predictions are compared with the labels parsed from the filenames.

In [ ]:
@torch.inference_mode()
def predict(model, dataset, device, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    predictions, targets = [], []
    for batch_images, batch_targets in DataLoader(dataset, batch_size=batch_size, shuffle=False):
        predictions.append(model(batch_images.to(device)).argmax(dim=1).cpu())
        targets.append(batch_targets)
    return torch.cat(predictions), torch.cat(targets)


test_predictions, test_targets = predict(test_model, combined_test_dataset, DEVICE)
digit_correct = test_predictions.eq(test_targets)  # [500, 5]
image_correct = digit_correct.all(dim=1)  # [500]
# (test set name, path) of every sample, in the order of combined_test_dataset
sample_info = [(name, path) for name, dataset in test_datasets.items() for path in dataset.image_paths]
print(f"per-digit accuracy {digit_correct.float().mean():.2%}, full-image accuracy {image_correct.float().mean():.2%}, "
      f"failures: {(~image_correct).sum().item()} / {len(image_correct)}")


def to_text(row):
    return "".join(str(int(value)) for value in row)


def show_predictions(indices, title, columns=4):
    rows = math.ceil(len(indices) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 2.4 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, index in zip(axes.flat, indices):
        index = int(index)
        set_name, path = sample_info[index]
        raw = decode_image(str(path), mode=ImageReadMode.RGB).permute(1, 2, 0).numpy()
        truth, prediction = to_text(test_targets[index]), to_text(test_predictions[index])
        marks = "".join("^" if t != p else " " for t, p in zip(truth, prediction))
        axis.imshow(raw)
        axis.set_title(f"{set_name} | GT {truth}\npred {prediction}  {marks}",
                       color="green" if truth == prediction else "red", fontsize=10, family="monospace")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


rng = np.random.default_rng(SEED)
correct_indices = np.flatnonzero(image_correct.numpy())
failure_indices = np.flatnonzero(~image_correct.numpy())
show_predictions(sorted(rng.choice(correct_indices, size=min(8, len(correct_indices)), replace=False)),
                 "Correct test predictions (green)")
# up to four failures from every test set
shown_failures = []
for set_name in test_datasets:
    shown_failures += [i for i in failure_indices if sample_info[i][0] == set_name][:4]
show_predictions(shown_failures, "Failure cases (red; ^ marks wrong positions)")

In [ ]:
set_names = list(test_datasets)
set_of_sample = np.array([sample_info[i][0] for i in range(len(sample_info))])
wrong = (~digit_correct).numpy()

fig, axes = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)
# (a) full-image accuracy per test set
accuracy = [image_correct.numpy()[set_of_sample == name].mean() * 100 for name in set_names]
bars = axes[0].bar(set_names, accuracy, color="tab:blue")
axes[0].bar_label(bars, fmt="%.0f")
axes[0].set(ylabel="full-image accuracy (%)", ylim=(0, 100), title="(a) Full-image accuracy per test set")
# (b) error rate per digit position and test set
x = np.arange(5)
for k, name in enumerate(set_names):
    axes[1].bar(x + (k - 2) * 0.16, wrong[set_of_sample == name].mean(axis=0) * 100, width=0.16, label=name)
axes[1].set_xticks(x, [f"position {p + 1}" for p in range(5)])
axes[1].set(ylabel="digit error rate (%)", title="(b) Errors by digit position")
axes[1].legend(fontsize=8)
# (c) number of wrong digits in failed images
wrong_per_failed = wrong[~image_correct.numpy()].sum(axis=1)
counts = Counter(wrong_per_failed.tolist())
axes[2].bar([str(k) for k in sorted(counts)], [counts[k] for k in sorted(counts)], color="tab:red")
axes[2].set(xlabel="wrong digits in a failed image", ylabel="images", title="(c) Severity of failures")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
plt.show()

pairs = Counter(
    (int(t), int(p))
    for t, p, w in zip(test_targets.flatten(), test_predictions.flatten(), wrong.flatten())
    if w
)
print("most frequent confusions (true -> predicted):",
      ", ".join(f"{t}->{p} ({n})" for (t, p), n in pairs.most_common(10)))

In [ ]:
# Layout check on test0 (black, unrotated digits on a clean background): locate the five digits by
# their column projection and test whether digit k lies in the k-th 50-px slot that our generator uses
# for every training image (slot k spans x in [5 + 50k, 5 + 50(k+1)) ).
def digit_column_spans(path, threshold=100):
    gray = np.asarray(Image.open(path).convert("L")).astype(int)[4:-4, 4:-4]  # drop the border
    columns = (gray < threshold).any(axis=0)
    spans, start = [], None
    for x, filled in enumerate(np.append(columns, False)):
        if filled and start is None:
            start = x
        elif not filled and start is not None:
            spans.append((start + 4, x + 4))
            start = None
    return spans


in_slot, out_of_slot = [], []
for index, (set_name, path) in enumerate(sample_info):
    if set_name != "test0":
        continue
    spans = digit_column_spans(path)
    if len(spans) != 5:  # touching digits cannot be separated by the projection
        continue
    centers = [(a + b) / 2 for a, b in spans]
    outside = any(not (5 + 50 * k <= c < 5 + 50 * (k + 1)) for k, c in enumerate(centers))
    (out_of_slot if outside else in_slot).append(index)

analysed = len(in_slot) + len(out_of_slot)
print(f"test0 images analysed: {analysed}; with >= 1 digit outside its training slot: {len(out_of_slot)} "
      f"({len(out_of_slot) / analysed:.0%})")
for name, group in (("all digits inside their slots", in_slot), (">= 1 digit outside its slot", out_of_slot)):
    if group:
        print(f"  {name:30s}: {len(group):3d} images, full-image accuracy {image_correct[group].float().mean():.1%}")

**Failure analysis.**

*Where the errors are.* The failures are not spread uniformly over the test data. `test4` — the set whose images look most like our training data (random colors, rotations, interference lines, pixel noise) — is solved at ≈ 90%, whereas the visually *easier* sets `test0`–`test3` (unrotated digits; `test0` even black digits on a clean background) are only ≈ 67–73% correct. The failures also differ in kind: in `test0`–`test3` a failed image contains on average about two wrong digits (e.g. `test0`: ≈ 68 wrong digits in ≈ 32 failed images, from 86.4% per-digit / 68% full-image accuracy), whereas in `test4` it is about one (≈ 12 wrong digits in ≈ 10 failed images). Several neighboring positions failing together indicates that the model assigns digits to the wrong positions rather than merely misreading an isolated digit (see panels b and c and the failure grid).

*Why: the digit layout differs from our generator.* Our generator places digit *k* inside the *k*-th fixed 50-px slot, so every training and validation image has five roughly evenly spaced digits. In the test images the digits are positioned much more freely: large gaps, two digits squeezed close together, and digits far from "their" slot. The layout check above makes this concrete for `test0`: a large fraction of its images contain at least one digit outside the slot it would occupy in our data — a configuration that never occurs in training — and the two groups can be compared directly. A network that has only seen evenly spaced digits learns position-specific features ("the third digit is around x ≈ 130"); when two digits fall into one slot or a slot is empty, it reads a neighboring digit for a position, which typically corrupts two adjacent positions at once. This is consistent with the ablations: augmentation with small random shifts/scales helps the test accuracy most, more data from the same generator does not, and the position-specific flatten head generalizes worse than global average pooling. The JPEG compression of the test images is a further, probably smaller, difference.

*Other patterns.* The failure grid and the confusion list above show the remaining, more ordinary error sources, such as digits that touch or overlap a neighbor and digits crossed by an interference line.

*How to fix it.* The validation set comes from the same generator as the training set and therefore did not reveal the problem (≈ 100% validation vs. 74% test). The most promising fixes are on the data side: generate digits with free left-to-right positions (random gaps, allowing close/overlapping digits, wider size range), optionally include unrotated/clean images, and stronger translation augmentation — then retrain the same ResNet-18.